In [22]:
import requests
from bs4 import BeautifulSoup
import json

url = "https://www.yapo.cl/chile-es/ajax/autos-usados"

headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "X-Requested-With": "XMLHttpRequest"
}

params = {
    "sort": "f_added",
    "dir": "desc",
    "regionslug": "region-metropolitana",
    "page": 1,
    "list": "category"
}

response = requests.get(url, headers=headers, params=params)
print(response.status_code)
print(response.json().keys())

data = response.json()
soup = BeautifulSoup(data['listing'], 'html.parser')

200
dict_keys(['url', 'listing', 'filter', 'tags', 'headlinenav'])


In [23]:
anuncios = soup.find_all('div', class_='d3-ad-tile')
print(f"Anuncios encontrados: {len(anuncios)}")

# Ver estructura completa del primer anuncio
print(anuncios[0].prettify()[:3000])

Anuncios encontrados: 30
<div class="d3-ad-tile d3-ads-grid__item d3-ad-tile--fullwidth d3-ad-tile--bordered" data-tracklisting="aHR0cHM6Ly93d3cueWFwby5jbC9jbGlja2hvdXNlX212cC5waHAvbGlzdHM/dD15b2ZaZDg0aUIwZDQwJTJGQmF1cEszeGxNSVRuRGdRMSUyQjZrUVJtUlRKdEdwdGJQbUpjRDYxTTBUdFg1Ujc3SzNFZE1RVnlCJTJCdVlud2ZzbXljS1p4JTJCd2FwYkhIOG0lMkYydE1PSXYwd1RtM1ZBY2FmMmk2Q05YQ2hHTE1SajVvJTJCSVNNa0hmaHVFNEttZzZpa0pFWHYxSHFOejdLSkJmNFhwTDJOSlNUUG9yRVA5aiUyRm9zY1QwaSUyQm43clVMVnhzM1ElMkI2JTJGVXJnc0I2cWZWJTJGenplRUtkR0ZwMVIlMkJFcEZnbnh5OTNwbkVkdU9pelBlZ1NTTnVZd2NYJTJGb1VySmFNdCUyQnZoZUx5bnpLR21oa0xtJTJGc0ZHV0RwUHF3USUyRlhkRGc2cG51VGdzeWYlMkJicFZnc21kUDlZTW5DQUZNMk9rNjNkJTJCUFZ3dEUlMkJSbllpdjR6dzA5UmxhUDNUWHBVc3JtU1Q0bXBCM1hwdGwwJTJGSU8lMkJUR0lkRjJtMmwyTTFpNDUlMkZKYnBvTGhjU3A5bXI5aFVaQTdhRXcxRVB2SEdNR0hYbUElM0QlM0Q=">
 <div class="d3-ad-tile__cover">
  <div class="d3-ad-tile__favorite-icon">
   <a class="tool-favorite" data-adid="31948237" data-isc2c="" data-price="13180.6" data-requirelogin="/account/user/logi

In [24]:
anuncio = anuncios[0]

# Título y URL
titulo = anuncio.find('a', class_='d3-ad-tile__description')['href']
print("URL:", titulo)

# Precio en UF
precio_uf = anuncio.find('a', class_='tool-favorite')['data-price']
print("Precio UF:", precio_uf)

# Detalles (año, km, combustible)
detalles = anuncio.find_all('li', class_='d3-ad-tile__details-item')
for d in detalles:
    print("Detalle:", d.get_text(strip=True))

# Nombre del anuncio completo
img = anuncio.find('img', class_='d3-ad-tile__photo')
print("Título:", img['alt'] if img else None)

# Vendedor
vendedor = anuncio.find('span', string=lambda t: t and t.strip())
print("Vendedor:", vendedor.get_text(strip=True) if vendedor else None)

# Fecha publicación
fecha = anuncio.find('span', class_='d3-ad-tile__date')
print("Fecha:", fecha.get_text(strip=True) if fecha else "No visible en listing")


URL: /autos-usados/peugeot-208-automatico-bluetooth/31948237
Precio UF: 13180.6
Detalle: 2022
Detalle: Bencina
Detalle: Automática
Detalle: 16000 km
Título: Peugeot 208 Automatico Bluetooth
Vendedor: Añadir a favoritos
Fecha: No visible en listing


In [25]:
# Buscar todos los elementos que contengan precio
for tag in anuncio.find_all(True):
    texto = tag.get_text(strip=True)
    clase = tag.get('class', [])
    if 'precio' in str(clase).lower() or 'price' in str(clase).lower():
        print(f"Clase: {clase} → Texto: {texto}")
vendedor = anuncio.find('div', class_='d3-ad-tile__seller')
print(vendedor.find('span').get_text(strip=True))


Clase: ['d3-ad-tile__price'] → Texto: $11.800.000-2%
Clase: ['d3-ad-tile__price-reduction'] → Texto: -2%
Clase: ['d3-icon', 'd3-ad-tile__price-reduction__icon'] → Texto: 
Esteban Dominguez


In [26]:
def extraer_titulo(anuncio):
    # Anuncio normal
    img = anuncio.find('img', class_='d3-ad-tile__photo')
    if img and img.get('alt'):
        return img['alt']
    
    # Anuncio destacado (feat-plat): el título está en el enlace
    titulo_tag = anuncio.find('h2', class_='d3-ad-tile__title')
    if titulo_tag:
        return titulo_tag.get_text(strip=True)
    
    # Fallback: extraer desde la URL
    url_tag = anuncio.find('a', class_='d3-ad-tile__description')
    if url_tag:
        slug = url_tag['href'].split('/')[-2]  # ej: "2015-peugeot-208-allure"
        return slug.replace('-', ' ').title()
    
    return None


In [27]:
import pandas as pd

def extraer_anuncio(anuncio):
    try:
        # URL e ID
        url_relativa = anuncio.find('a', class_='d3-ad-tile__description')['href']
        url = f"https://www.yapo.cl{url_relativa}"
        id_anuncio = url_relativa.split('/')[-1]

        # Título
        img = anuncio.find('img', class_='d3-ad-tile__photo')
        titulo = extraer_titulo(anuncio)


        # Precio CLP
        precio_tag = anuncio.find(class_='d3-ad-tile__price')
        precio_raw = precio_tag.get_text(strip=True) if precio_tag else None

        # Vendedor
        vendedor_tag = anuncio.find('div', class_='d3-ad-tile__seller')
        vendedor = vendedor_tag.find('span').get_text(strip=True) if vendedor_tag else None

        # Detalles: año, combustible, transmisión, km
        detalles = [d.get_text(strip=True) for d in anuncio.find_all('li', class_='d3-ad-tile__details-item')]
        año        = detalles[0] if len(detalles) > 0 else None
        combustible = detalles[1] if len(detalles) > 1 else None
        transmision = detalles[2] if len(detalles) > 2 else None
        kilometraje = detalles[3] if len(detalles) > 3 else None

        return {
            'id': id_anuncio,
            'titulo': titulo,
            'precio_clp': precio_raw,
            'año': año,
            'combustible': combustible,
            'transmision': transmision,
            'kilometraje': kilometraje,
            'vendedor': vendedor,
            'url': url
        }
    except Exception as e:
        return None

# Extraer los 30 anuncios
registros = [extraer_anuncio(a) for a in anuncios]
registros = [r for r in registros if r is not None]

df = pd.DataFrame(registros)
print(f"Anuncios extraídos: {len(df)}")
df.head()


Anuncios extraídos: 30


,id,titulo,precio_clp,año,combustible,transmision,kilometraje,vendedor,url
0,31948237,Peugeot 208 Automatico Bluetooth,$11.800.000-2%,2022,Bencina,Automática,16000 km,Esteban Dominguez,https://www.yapo.cl/autos-usados/peugeot-208-a...
1,32080232,XTRAIL AÑO 2009,$4.990.000,2009,Bencina,Automática,200000 km,Jaime Valdes,https://www.yapo.cl/autos-usados/xtrail-ano-20...
2,32080154,Furgón Toyota Hiace 2009 Petrolero,$5.990.000,2009,Diesel,Manual,289000 km,Benjamin,https://www.yapo.cl/autos-usados/furgon-toyota...
3,32080097,Peugeot partner año 2022 L2,$12.200.000,2022,Diesel,Manual,500000 km,wilber,https://www.yapo.cl/autos-usados/peugeot-partn...
4,32080073,OFERTA por viaje,$9.500.000,2023,Bencina,Manual,NaN,Julio Vera,https://www.yapo.cl/autos-usados/oferta-por-vi...


In [28]:
url_anuncio = "https://www.yapo.cl" + "/autos-usados/2015-peugeot-208-allure-e-hdi-1-6/32079630"

resp = requests.get(url_anuncio, headers=headers)
soup_detalle = BeautifulSoup(resp.text, 'html.parser')

# Imprimir todas las clases únicas del anuncio individual
clases = set()
for tag in soup_detalle.find_all(True):
    for clase in tag.get('class', []):
        clases.add(clase)

# Filtrar las que parecen relevantes (ignorar clases genéricas de diseño)
relevantes = [c for c in sorted(clases) if any(x in c for x in 
    ['detail', 'info', 'data', 'date', 'time', 'price', 'feature', 
     'spec', 'attr', 'param', 'field', 'region', 'location', 'seller'])]

print('\n'.join(relevantes))



adaptor-breadcrumb-detailpager
contact_info
d3-ad-tile__location-icon
d3-property-about__details
d3-property-breadcrumbcatandregion
d3-property-contact__address-details
d3-property-contact__detail
d3-property-contact__detail--seals
d3-property-details
d3-property-details__content
d3-property-details__detail
d3-property-details__detail-label
d3-property-details__title
d3-property-info
d3-property-info__actions
d3-property-info__basic
d3-property-info__contact
d3-property-info__header
d3-property-info__header-data
d3-property-info__images
d3-property-info__images--container
d3-property-info__price
d3-property-info__title
d3-property-info__title--text-black
d3-property-info__vendor
d3-property-insight__attribute
d3-property-insight__attribute--bordered
d3-property-insight__attribute-details
d3-property-insight__attribute-icon
d3-property-insight__attribute-icon--brand-color
d3-property-insight__attribute-title
d3-property-insight__attribute-value
d3-textfield
d3-textfield__field
d3-textfi

In [29]:
# Ver todos los detalles del anuncio individual
print("=== DETALLES ===")
for tag in soup_detalle.find_all(class_='d3-property-details__detail'):
    print(tag.get_text(strip=True))

print("\n=== INSIGHTS (atributos) ===")
for tag in soup_detalle.find_all(class_='d3-property-insight__attribute'):
    titulo = tag.find(class_='d3-property-insight__attribute-title')
    valor = tag.find(class_='d3-property-insight__attribute-value')
    if titulo and valor:
        print(f"{titulo.get_text(strip=True)}: {valor.get_text(strip=True)}")

print("\n=== HEADER DATA (fecha?) ===")
for tag in soup_detalle.find_all(class_='d3-property-info__header-data'):
    print(tag.get_text(strip=True))

print("\n=== UBICACIÓN ===")
for tag in soup_detalle.find_all(class_='d3-property-contact__address-details'):
    print(tag.get_text(strip=True))


=== DETALLES ===
La Reina
05/03/2026
Automática
Peugeot

=== INSIGHTS (atributos) ===
Marca: Peugeot
Modelo: 208
Precio: $9.280.000
Año: 2015
Kilómetros: 54'997
Combustible: Diesel

=== HEADER DATA (fecha?) ===
La Reina05/03/2026

=== UBICACIÓN ===
Pedro PelayoAutomoviles pelayo ltdaAVDA OSSA 315 LA REINA -


In [30]:
import requests
import time
from bs4 import BeautifulSoup
import pandas as pd

# Configuración
API_URL = "https://www.yapo.cl/chile-es/ajax/autos-usados"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "X-Requested-With": "XMLHttpRequest"
}

REGIONES = {
    "region-metropolitana": 3000,
    "valparaiso":           1000,
    "biobio":               600,
    "los-lagos":            200,
    "antofagasta":          200
}

def extraer_titulo(anuncio):
    img = anuncio.find('img', class_='d3-ad-tile__photo')
    if img and img.get('alt'):
        return img['alt']
    titulo_tag = anuncio.find('h2', class_='d3-ad-tile__title')
    if titulo_tag:
        return titulo_tag.get_text(strip=True)
    url_tag = anuncio.find('a', class_='d3-ad-tile__description')
    if url_tag:
        slug = url_tag['href'].split('/')[-2]
        return slug.replace('-', ' ').title()
    return None

def extraer_anuncio(anuncio, region):
    try:
        url_relativa = anuncio.find('a', class_='d3-ad-tile__description')['href']
        url = f"https://www.yapo.cl{url_relativa}"
        id_anuncio = url_relativa.split('/')[-1]
        titulo = extraer_titulo(anuncio)
        precio_tag = anuncio.find(class_='d3-ad-tile__price')
        precio_raw = precio_tag.get_text(strip=True) if precio_tag else None
        vendedor_tag = anuncio.find('div', class_='d3-ad-tile__seller')
        vendedor = vendedor_tag.find('span').get_text(strip=True) if vendedor_tag else None
        detalles = [d.get_text(strip=True) for d in anuncio.find_all('li', class_='d3-ad-tile__details-item')]
        es_destacado = 'd3-ad-tile--feat-plat' in anuncio.get('class', [])

        return {
            'id':           id_anuncio,
            'titulo':       titulo,
            'precio_clp':   precio_raw,
            'año':          detalles[0] if len(detalles) > 0 else None,
            'combustible':  detalles[1] if len(detalles) > 1 else None,
            'transmision':  detalles[2] if len(detalles) > 2 else None,
            'kilometraje':  detalles[3] if len(detalles) > 3 else None,
            'vendedor':     vendedor,
            'region':       region,
            'destacado':    es_destacado,
            'url':          url
        }
    except Exception as e:
        return None

def scrape_pagina(region, pagina=1):
    params = {
        "sort": "f_added",
        "dir":  "desc",
        "regionslug": region,
        "page": pagina,
        "list": "category"
    }
    response = requests.get(API_URL, headers=HEADERS, params=params)
    if response.status_code != 200:
        print(f"Error {response.status_code} en {region} página {pagina}")
        return []

    soup = BeautifulSoup(response.json()['listing'], 'html.parser')
    anuncios = soup.find_all('div', class_='d3-ad-tile')
    return [r for r in [extraer_anuncio(a, region) for a in anuncios] if r is not None]

# ── PRUEBA: solo página 1 de cada región ──
resultados = []
for region in REGIONES:
    registros = scrape_pagina(region, pagina=1)
    resultados.extend(registros)
    print(f"{region}: {len(registros)} anuncios")
    time.sleep(2)

df_prueba = pd.DataFrame(resultados)
print(f"\nTotal: {len(df_prueba)} anuncios")
print(df_prueba[['region', 'titulo', 'precio_clp', 'año', 'kilometraje']].head(10))


region-metropolitana: 30 anuncios
valparaiso: 30 anuncios
biobio: 30 anuncios
los-lagos: 30 anuncios
antofagasta: 30 anuncios

Total: 150 anuncios
                 region                                titulo      precio_clp  \
0  region-metropolitana      Peugeot 208 Automatico Bluetooth  $11.800.000-2%   
1  region-metropolitana                       XTRAIL AÑO 2009      $4.990.000   
2  region-metropolitana    Furgón Toyota Hiace 2009 Petrolero      $5.990.000   
3  region-metropolitana           Peugeot partner año 2022 L2     $12.200.000   
4  region-metropolitana                      OFERTA por viaje      $9.500.000   
5  region-metropolitana       VENDO TOYOTA YARIS GLI 1.5 2021     $13.000.000   
6  region-metropolitana          FORD RANGER 2.5 XLT AÑO 2016     $10.900.000   
7  region-metropolitana                   BMW 320 IA AÑO 2010      $5.780.000   
8  region-metropolitana  focos neblineros peugeot 308 2012 t7         $35.000   
9  region-metropolitana     2015 Peugeot 20

In [ ]:
df_prueba

In [19]:
import time

def scrape_detalle(url, id_anuncio):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        if resp.status_code != 200:
            return None

        soup = BeautifulSoup(resp.text, 'html.parser')

        # Fecha y comuna (vienen juntos en header-data)
        header_data = soup.find(class_='d3-property-info__header-data')
        comuna, fecha_publicacion = None, None
        if header_data:
            texto = header_data.get_text(separator='|', strip=True).split('|')
            comuna = texto[0].strip() if len(texto) > 0 else None
            fecha_publicacion = texto[1].strip() if len(texto) > 1 else None

        # Marca y modelo desde insights
        marca, modelo = None, None
        for tag in soup.find_all(class_='d3-property-insight__attribute'):
            titulo_tag = tag.find(class_='d3-property-insight__attribute-title')
            valor_tag  = tag.find(class_='d3-property-insight__attribute-value')
            if not titulo_tag or not valor_tag:
                continue
            titulo_txt = titulo_tag.get_text(strip=True).lower()
            valor_txt  = valor_tag.get_text(strip=True)
            if 'marca' in titulo_txt:
                marca = valor_txt
            elif 'modelo' in titulo_txt:
                modelo = valor_txt

        # Empresa y dirección del vendedor
        empresa, direccion = None, None
        vendedor_tag = soup.find(class_='d3-property-contact__address-details')
        if vendedor_tag:
            textos = [t.strip() for t in vendedor_tag.get_text(separator='|').split('|') if t.strip()]
            empresa   = textos[1] if len(textos) > 1 else None
            direccion = textos[2] if len(textos) > 2 else None

        # Tipo de vendedor (particular o profesional)
        tipo_vendedor = None
        seal = soup.find('img', alt=lambda x: x and x.strip() in ['Profesional', 'Particular'])
        if seal:
            tipo_vendedor = seal['alt'].lower()

        return {
            'id':                 id_anuncio,
            'fecha_publicacion':  fecha_publicacion,
            'comuna':             comuna,
            'marca':              marca,
            'modelo':             modelo,
            'empresa':            empresa,
            'direccion':          direccion,
            'tipo_vendedor':      tipo_vendedor
        }

    except Exception as e:
        print(f"Error en {url}: {e}")
        return None


# ── PRUEBA con 3 anuncios del df_prueba ──
muestra = df_prueba[['id', 'url']].head(3)

resultados_detalle = []
for _, row in muestra.iterrows():
    resultado = scrape_detalle(row['url'], row['id'])
    print(resultado)
    resultados_detalle.append(resultado)
    time.sleep(2)

df_detalle_prueba = pd.DataFrame([r for r in resultados_detalle if r])
print(df_detalle_prueba)


{'id': '32079793', 'fecha_publicacion': '05/03/2026', 'comuna': 'Ñuñoa', 'marca': 'BMW', 'modelo': '320', 'empresa': 'Diagonal Autos', 'direccion': 'Presidente Jose Batlle y Ordoñez 3701, Ñuñoa, Santiago -', 'tipo_vendedor': None}
{'id': '32079731', 'fecha_publicacion': '05/03/2026', 'comuna': 'Maipú', 'marca': 'Peugeot', 'modelo': '308', 'empresa': None, 'direccion': None, 'tipo_vendedor': None}
{'id': '32079630', 'fecha_publicacion': '05/03/2026', 'comuna': 'La Reina', 'marca': 'Peugeot', 'modelo': '208', 'empresa': 'Automoviles pelayo ltda', 'direccion': 'AVDA OSSA 315 LA REINA -', 'tipo_vendedor': None}
         id fecha_publicacion    comuna    marca modelo  \
0  32079793        05/03/2026     Ñuñoa      BMW    320   
1  32079731        05/03/2026     Maipú  Peugeot    308   
2  32079630        05/03/2026  La Reina  Peugeot    208   

                   empresa                                          direccion  \
0           Diagonal Autos  Presidente Jose Batlle y Ordoñez 3701, 